In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
import json

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [71]:
import os
import json


Phase 1 – Dataset Loading

In [72]:
#1
base_path = "/kaggle/input/2017-2017"

print("Base dataset folder:")
print(base_path)

print("\nSubfolders:")
for item in os.listdir(base_path):
    item_path = os.path.join(base_path, item)
    if os.path.isdir(item_path):
        print("  └──", item)


Base dataset folder:
/kaggle/input/2017-2017

Subfolders:
  └── image_info_unlabeled2017
  └── val2017
  └── annotations_trainval2017
  └── image_info_test2017
  └── test2017
  └── train2017
  └── unlabeled2017


In [75]:
#2 (FINAL - CORRECT FOR YOUR DATASET)
import os

BASE_PATH = "/kaggle/input/2017-2017"

TRAIN_IMAGES_PATH = os.path.join(BASE_PATH, "train2017")
VAL_IMAGES_PATH   = os.path.join(BASE_PATH, "val2017")

ANNOTATIONS_PATH  = os.path.join(BASE_PATH, "annotations_trainval2017", "annotations")

TRAIN_CAPTIONS_FILE = os.path.join(ANNOTATIONS_PATH, "captions_train2017.json")
VAL_CAPTIONS_FILE   = os.path.join(ANNOTATIONS_PATH, "captions_val2017.json")

print("TRAIN_IMAGES_PATH exists:", os.path.exists(TRAIN_IMAGES_PATH))
print("VAL_IMAGES_PATH exists  :", os.path.exists(VAL_IMAGES_PATH))
print("Train captions exists   :", os.path.exists(TRAIN_CAPTIONS_FILE))
print("Val captions exists     :", os.path.exists(VAL_CAPTIONS_FILE))

ANN_ROOT exists: True

Top-level files/folders in annotations_trainval2017:
['annotations']

All JSON files under annotations_trainval2017:
Found: 6
/kaggle/input/2017-2017/annotations_trainval2017/annotations/captions_train2017.json
/kaggle/input/2017-2017/annotations_trainval2017/annotations/captions_val2017.json
/kaggle/input/2017-2017/annotations_trainval2017/annotations/instances_train2017.json
/kaggle/input/2017-2017/annotations_trainval2017/annotations/instances_val2017.json
/kaggle/input/2017-2017/annotations_trainval2017/annotations/person_keypoints_train2017.json
/kaggle/input/2017-2017/annotations_trainval2017/annotations/person_keypoints_val2017.json

Train caption guesses: ['/kaggle/input/2017-2017/annotations_trainval2017/annotations/captions_train2017.json']
Val caption guesses: ['/kaggle/input/2017-2017/annotations_trainval2017/annotations/captions_val2017.json']


PHASE 2: DATA UNDERSTANDING

STEP 1: Load the Training Captions JSON

In [41]:
#3
import json

with open(TRAIN_CAPTIONS_FILE, "r") as f:
    train_data = json.load(f)

print("Top-level keys in training annotation file:")
print(train_data.keys())


Top-level keys in training annotation file:
dict_keys(['info', 'licenses', 'images', 'annotations'])


STEP 3: Inspect Images Metadata

In [42]:
#4
print("\nSample image entry:")
train_data["images"][0]



Sample image entry:


{'license': 3,
 'file_name': '000000391895.jpg',
 'coco_url': 'http://images.cocodataset.org/train2017/000000391895.jpg',
 'height': 360,
 'width': 640,
 'date_captured': '2013-11-14 11:18:45',
 'flickr_url': 'http://farm9.staticflickr.com/8186/8119368305_4e622c8349_z.jpg',
 'id': 391895}

In [43]:
# 5
with open(VAL_CAPTIONS_FILE, "r") as f:
    val_data = json.load(f)

print("Number of training images:", len(train_data["images"]))
print("Number of validation images:", len(val_data["images"]))


Number of training images: 118287
Number of validation images: 5000


STEP 4.1: Count Total Captions

In [44]:
#6
print("Number of captions:", len(train_data["annotations"]))
print("Sample caption entry:")
train_data["annotations"][0]



Number of captions: 591753
Sample caption entry:


{'image_id': 203564,
 'id': 37,
 'caption': 'A bicycle replica with a clock as the front wheel.'}

STEP 4.3: Verify Image → Multiple Captions

In [45]:
#7
from collections import defaultdict

image_caption_count = defaultdict(int)

for ann in train_data["annotations"]:
    image_caption_count[ann["image_id"]] += 1

# Check a few images
list(image_caption_count.items())[:5]


[(203564, 5), (322141, 5), (16977, 5), (106140, 5), (571635, 5)]

STEP 5: IMAGE → CAPTIONS MAPPING

STEP 5.1: Map image_id → file_name

In [46]:
#8
# Create image_id → filename mapping
image_id_to_filename = {}

for img in train_data["images"]:
    image_id_to_filename[img["id"]] = img["file_name"]

print("Total image IDs mapped:", len(image_id_to_filename))


Total image IDs mapped: 118287


STEP 5.2: Build filename → captions Dictionary

In [47]:
#9
from collections import defaultdict

image_to_captions = defaultdict(list)

for ann in train_data["annotations"]:
    image_id = ann["image_id"]
    caption = ann["caption"]
    filename = image_id_to_filename[image_id]
    image_to_captions[filename].append(caption)

print("Total images with captions:", len(image_to_captions))


Total images with captions: 118287


STEP 5.3: Inspect One Image’s Captions

In [ ]:
#10
sample_image = list(image_to_captions.keys())[0]

print("Image filename:", sample_image)
print("Captions:")
for cap in image_to_captions[sample_image]:
    print("-", cap)


EDA STEP 1: SAMPLE IMAGES WITH CAPTIONS (MANDATORY)

In [ ]:
#11
import matplotlib.pyplot as plt

# Dataset info
splits = ['Train', 'Validation']
num_images = [118287, 5000]
num_captions = [118287*5, 5000*5]  # 5 captions per image

# Create bar chart
bar_width = 0.35
x = range(len(splits))

plt.figure(figsize=(8,5))

# Bars for images
plt.bar([i - bar_width/2 for i in x], num_images, width=bar_width, label='Images', color='skyblue')

# Bars for captions
plt.bar([i + bar_width/2 for i in x], num_captions, width=bar_width, label='Captions', color='orange')

# Labels and title
plt.xticks(x, splits)
plt.ylabel('Count')
plt.title('Number of Images vs Captions per Split')
plt.legend()
plt.show()


In [ ]:
#12
import matplotlib.pyplot as plt
import requests
from PIL import Image
from io import BytesIO
import random
import textwrap

# Pick 2 random images
sample_images = random.sample(train_data['images'], 2)

plt.figure(figsize=(14, 6))

for i, img_info in enumerate(sample_images):
    img_id = img_info['id']
    img_url = img_info['coco_url']

    # Get ALL captions for this image (5 captions)
    captions = [
        ann['caption']
        for ann in train_data['annotations']
        if ann['image_id'] == img_id
    ]

    # Load image from URL
    response = requests.get(img_url)
    image = Image.open(BytesIO(response.content))

    # Plot image
    plt.subplot(1, 2, i + 1)
    plt.imshow(image)
    plt.axis("off")

    # Format captions nicely (numbered + wrapped)
    caption_text = "\n".join(
        [f"{j+1}. {textwrap.fill(cap, 45)}" for j, cap in enumerate(captions)]
    )

    plt.title(caption_text, fontsize=9, loc='left')



plt.show()


PHASE 3: PREPROCESSING

STEP 3.1.1: Text Cleaning Function
What this does

Lowercase

Remove punctuation

Remove extra spaces

In [49]:
#13
import re

def clean_caption(caption):
    caption = caption.lower()
    caption = re.sub(r"[^a-z\s]", "", caption)
    caption = re.sub(r"\s+", " ", caption).strip()
    return caption


STEP 3.1.2: Add <start> and <end> Tokens

In [51]:
#14
def add_tokens(caption):
    return "<start> " + caption + " <end>"


STEP 3.1.3: Apply Cleaning to All Training Captions

In [52]:
#15
all_captions = []

for captions in image_to_captions.values():
    for cap in captions:
        cap = clean_caption(cap)
        cap = add_tokens(cap)
        all_captions.append(cap)

print("Total processed captions:", len(all_captions))
print("Sample caption:", all_captions[0])

Total processed captions: 591753
Sample caption: <start> a bicycle replica with a clock as the front wheel <end>


PHASE 3.2: TOKENIZATION & VOCABULARY

STEP 3.2.1: Tokenize Captions

In [53]:
#16
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(
    num_words=10000,
    oov_token="<unk>",
    filters=""
)

tokenizer.fit_on_texts(all_captions)


STEP 3.2.2: Vocabulary Size

In [54]:
#17
# Limit vocabulary to the top 10,000 most frequent words
vocab_size = min(10000, len(tokenizer.word_index) + 1)
print("Vocabulary size:", vocab_size)


Vocabulary size: 10000


STEP 3.2.3: Convert Captions to Sequences

What it does (very brief):

texts_to_sequences → turns words into numbers

max_length → finds the longest caption

pad_sequences → adds zeros so all captions have the same length

Why needed:
Neural networks require fixed-length inputs.

In [55]:
#18
from tensorflow.keras.preprocessing.sequence import pad_sequences

sequences = tokenizer.texts_to_sequences(all_captions)

max_length = max(len(seq) for seq in sequences)
print("Maximum caption length:", max_length)

padded_sequences = pad_sequences(
    sequences,
    maxlen=max_length,
    padding="post"
)


Maximum caption length: 51


In [56]:
#savepoint 
import pickle

# Save tokenizer
pickle.dump(tokenizer, open("/kaggle/working/tokenizer.pkl", "wb"))

# Save important config
config = {
    "vocab_size": vocab_size,
    "max_length": max_length
}

pickle.dump(config, open("/kaggle/working/config.pkl", "wb"))

print("Tokenizer and config saved successfully.")


Tokenizer and config saved successfully.


PHASE 3.3: IMAGE PREPROCESSING (ResNet-101)

In [57]:
#19
from tensorflow.keras.applications.resnet import ResNet101, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model

base_model = ResNet101(
    weights="imagenet",
    include_top=False
)

encoder = Model(
    inputs=base_model.input,
    outputs=base_model.output
)

encoder.trainable = False


STEP 3.3.2: Image Preprocessing Function

In [63]:
#20
import numpy as np
import os

def preprocess_image(image_path):
    img = load_img(image_path, target_size=(224, 224))
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)
    return img


In [ ]:
#21 (SAFE VERSION)

import matplotlib.pyplot as plt
from collections import Counter

# ✅ Use different variable name (IMPORTANT)
sample_image_to_captions = {
    "000000203564.jpg": [
        "A bicycle replica with a clock as the front wheel.",
        "The bike has a clock as a tire.",
        "A black metal bicycle with a clock inside the front wheel.",
        "A bicycle figurine in which the front wheel is replaced with a clock",
        "A clock with the appearance of the wheel of a bicycle"
    ]
}

sample_image = list(sample_image_to_captions.keys())[0]
sample_captions = sample_image_to_captions[sample_image]

# Caption length plot
caption_lengths = [len(cap.split()) for cap in sample_captions]

plt.figure(figsize=(6,4))
plt.bar(range(len(sample_captions)), caption_lengths)
plt.show()

# Word frequency plot
words = [word.lower().strip(".,") for cap in sample_captions for word in cap.split()]
word_counts = Counter(words)
top_words = word_counts.most_common(10)
words_list, counts_list = zip(*top_words)

plt.figure(figsize=(6,4))
plt.bar(words_list, counts_list)
plt.show()

In [ ]:
#22
import os
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Base dataset folder
image_folder = "/kaggle/input/2017-2017/train2017"

# Walk recursively to find the first .jpg file
image_path = None
for root, dirs, files in os.walk(image_folder):
    for f in files:
        if f.endswith(".jpg"):
            image_path = os.path.join(root, f)
            break
    if image_path:
        break

if image_path is None:
    raise ValueError("No .jpg images found in the dataset!")

print("Using image:", image_path)

# Open the original image and convert to RGB
img_original = Image.open(image_path).convert("RGB")

# Resize to 224x224
img_resized = img_original.resize((224, 224))

# Convert to numpy arrays for plotting
orig_array = np.array(img_original)
resized_array = np.array(img_resized)

# Plot side by side
plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(orig_array)
plt.title(f"Original: {img_original.size[0]}x{img_original.size[1]}")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(resized_array)
plt.title(f"Resized: 224x224")
plt.axis('off')

plt.tight_layout()
plt.show()


In [69]:
#23
import os
import pickle
import numpy as np
from tqdm import tqdm

TRAIN_IMG_DIR = "/kaggle/input/2017-2017/train2017"

image_features = {}
image_list = list(image_to_captions.keys())[:5000]

for filename in tqdm(image_list):
    img_path = os.path.join(TRAIN_IMG_DIR, filename)

    if not os.path.exists(img_path):
        continue

    img = preprocess_image(img_path)
    feature = encoder.predict(img, verbose=0)  # (1,7,7,2048)
    feature = feature.reshape(49, 2048)
    image_features[filename] = feature

with open("image_features.pkl", "wb") as f:
    pickle.dump(image_features, f)

print("Features saved. Total:", len(image_features))

100%|██████████| 5000/5000 [00:00<00:00, 195000.46it/s]

Features saved. Total: 0
